# Mini-lab — Skillbox / toolbox retrieval

Goal: see how an agent with many tools avoids context-window collapse by retrieving only the top-k relevant tools and skills for the current turn.


## 0. Why this pattern exists

If an agent has 50 tools, pasting every JSON schema into every prompt is wasteful and confusing. The toolbox stores tool definitions as vector-searchable rows. The skillbox does the same for prose playbooks: the prompt gets a compact manifest, and the full skill body is loaded only when needed.


In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

import json
from pathlib import Path

from agentic_memory_workshop.connection import load_env, connect
from agentic_memory_workshop.toolbox import upsert_tool, retrieve_tools, count_tools
from agentic_memory_workshop.skillbox import ingest_markdown_file, build_skill_manifest, get_skill, count_skills

load_env('../.env')
ROOT = Path('..').resolve()
connection = connect()
print('Connected to Oracle AI Database')


## 1. Ingest ten sample tools

Each tool has a name, description, and JSON parameters. `upsert_tool` stores the row and asks Oracle to compute `VECTOR_EMBEDDING(...)` in the database.


In [ ]:
tools = json.loads((ROOT / 'samples/tools.json').read_text())
try:
    for tool in tools:
        upsert_tool(connection, tool['name'], tool['description'], tool['parameters'], source='sample')
    print(f'Tools in registry: {count_tools(connection)}')
except Exception as exc:
    print('Tool ingest failed. Most common cause: the in-database ONNX model is not loaded yet.')
    print(type(exc).__name__, exc)
    raise


## 2. Retrieve only the relevant tools

Change the query and watch the top-k list shift. This is what goes into the model's active prompt instead of all tool schemas.


In [ ]:
query = 'The user wants to cancel a reservation and asks whether they can get a refund.'
for tool in retrieve_tools(connection, query, k=4):
    print(f"{tool['distance']:.3f}  {tool['name']} — {tool['description']}")


## 3. Ingest three prose skills

A skill is a markdown playbook. The agent sees only the short manifest until it calls `load_skill(name)`.


In [ ]:
for path in sorted((ROOT / 'samples/skills').glob('*.md')):
    info = ingest_markdown_file(connection, path)
    print('ingested', info)
print(f'Skills in registry: {count_skills(connection)}')


## 4. Build the prompt manifest

This compact manifest is safe to paste into the prompt. The full markdown body stays out of context until the agent explicitly loads it.


In [ ]:
manifest = build_skill_manifest(connection, query, k=2)
print(manifest)


## 5. Load one full skill body

This simulates the agent choosing a skill from the manifest and calling `load_skill(name)`.


In [ ]:
first_skill_name = manifest.split('\n')[1].split(':', 1)[0].removeprefix('- ').strip()
skill = get_skill(connection, first_skill_name)
print(skill['body'] if skill else 'No skill found')


## 6. Scale up: a conversation with Grok on a 50-tool, 50-skill registry

Ten tools and three skills are a toy. Here the registry grows to 50 of each, and a four-turn conversation shifts task every turn. Watch three things: the tools and skills loaded change with the task, the focused prompt grows only because the history grows, and the token count of a naive prompt that pastes the whole registry stays far above it. Token counts are estimates from `litellm.token_counter`.

In [ ]:
from agentic_memory_workshop.context_window import registry_snapshot

catalog_tools = json.loads((ROOT / 'samples/catalog/tools.json').read_text())
for tool in catalog_tools:
    upsert_tool(connection, tool['name'], tool['description'], tool['parameters'], source='catalog')
catalog_skills = sorted((ROOT / 'samples/catalog/skills').glob('*.md'))
for path in catalog_skills:
    ingest_markdown_file(connection, path)

print(f'Ingested {len(catalog_tools)} catalog tools and {len(catalog_skills)} catalog skills')
print(f'Registry now holds {count_tools(connection)} tools and {count_skills(connection)} skills')

snapshot = registry_snapshot(connection)
print(f"Pasting every tool schema takes about {snapshot['tools_tokens']:,} tokens (estimated)")
print(f"Pasting every skill body takes about {snapshot['skills_tokens']:,} tokens (estimated)")
print(f"Whole registry in one prompt: about {snapshot['tools_tokens'] + snapshot['skills_tokens']:,} tokens (estimated)")

In [ ]:
from datetime import date

from agentic_memory_workshop.context_window import (
    assemble_messages, breakdown, changes, format_breakdown, naive_total, offload_pct,
    print_summary, prompt_tokens_reported, render_skills, render_tools,
)
from agentic_memory_workshop.llm import chat_completion, llm_status

# Edit these four messages and re-run: the task shifts every turn.
USER_MESSAGES = [
    'We are 6 people on Friday evening and need a quiet place to talk. Two of us are vegan.',
    'Book it for 7pm and text the confirmation to +1 555 0123.',
    'Our flight is delayed and we will miss the booking. Can I cancel and keep the deposit?',
    'Arrange a ride from the airport to the hotel tonight. Flight BA117, 3 passengers.',
]
SYSTEM_PROMPT = (
    f'Today is {date.today():%A %d %B %Y}. You are a restaurant concierge agent. '
    'Use only the tools listed in this prompt. '
    'Name the tool and the arguments you would call next. Answer in at most 3 sentences.'
)

llm_ready, llm_detail = llm_status()
if not llm_ready:
    print(f'Skipped the Grok conversation: {llm_detail}')
else:
    history, previous_tools, rows = [], [], []
    for turn, message in enumerate(USER_MESSAGES, 1):
        # 1. Retrieve: top tools, a skill manifest, and the full body of the best skill.
        tools = retrieve_tools(connection, message, k=4)
        manifest = build_skill_manifest(connection, message, k=3)
        manifest_names = [line.split(':', 1)[0].removeprefix('- ').strip() for line in manifest.split('\n')[1:]]
        skill = get_skill(connection, manifest_names[0])

        # 2. Assemble the focused prompt.
        tools_text = render_tools(tools)
        skills_text = render_skills(manifest, skill)
        messages = assemble_messages(SYSTEM_PROMPT, tools_text, skills_text, history, message)

        # 3. Call Grok.
        response = chat_completion(messages)
        reply = response.choices[0].message.content or ''

        # 4. Measure: focused prompt vs pasting the whole registry.
        parts = breakdown(SYSTEM_PROMPT, tools_text, skills_text, history, message)
        naive = naive_total(snapshot, SYSTEM_PROMPT, history, message)
        reported = prompt_tokens_reported(response)
        tool_names = [t['name'] for t in tools]

        print(f'=== Turn {turn} ===')
        print(f'User: {message}')
        print(f"Tools loaded: {', '.join(tool_names)}")
        print(f'  {changes(previous_tools, tool_names)}')
        print(f"Skills in manifest: {', '.join(manifest_names)}")
        print(f"Skill loaded: {skill['name']}")
        print(f'Context window: {format_breakdown(parts)}')
        if reported is not None:
            print(f'Grok reported {reported:,} prompt tokens')
        print(f"Naive prompt (all {len(snapshot['tools'])} tools + all {len(snapshot['skills'])} skills): {naive:,} tokens (estimated)")
        print(f"Offloaded: {naive - parts['total']:,} tokens ({offload_pct(parts['total'], naive):.1f}%)")
        print(f'Grok: {reply}\n')

        history += [{'role': 'user', 'content': message}, {'role': 'assistant', 'content': reply}]
        previous_tools = tool_names
        rows.append({'turn': turn, 'focused': parts['total'], 'naive': naive})

    print_summary(rows)